<a href="https://colab.research.google.com/github/soberaf2359/PROGRAMA-DELFIN/blob/main/C%C3%B3digo%20G2%20a%20Semana%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def preprocesar_rndc_grupo2(ruta_archivo):
    """
    Ejecuta el pipeline de limpieza (Semana 1) para el análisis de sesgo de flota
    y precarización tarifaria en el ecosistema de datos logísticos.
    """
    print("=== INICIANDO PROTOCOLO DE LIMPIEZA RNDC - GRUPO 2 ===")

    # 1. Carga del conjunto de datos de microdatos reales
    try:
        # Se asegura la lectura de datos a escala masiva
        df = pd.read_csv(ruta_archivo, sep=',', quotechar='"', low_memory=False)
        print(f"Datos originales cargados: {df.shape[0]} registros.")
    except Exception as e:
        return f"Error al cargar el archivo: {e}"

    # 2. PROTOCOLO DE LIMPIEZA: Corrección Temporal
    # Solución del desfase histórico de 1899 en las marcas de tiempo de origen
    if 'FECHASALIDACARGUE' in df.columns and 'HORA_SALIDA_CARGUE' in df.columns:
        fecha_real = pd.to_datetime(df['FECHASALIDACARGUE'], errors='coerce').dt.date
        hora_real = pd.to_datetime(df['HORA_SALIDA_CARGUE'], errors='coerce').dt.time
        df['FECHA_HORA_SALIDA_CORREGIDA'] = pd.to_datetime(fecha_real.astype(str) + ' ' + hora_real.astype(str), errors='coerce')
        print("✓ Corrección Temporal completada (Desfase de 1899 solucionado).")

    # 3. PROTOCOLO DE LIMPIEZA: Normalización Financiera
    # Limpieza de caracteres no numéricos en 'valor_pactado' y 'valor_pagado'
    columnas_financieras = ['VALOR_PACTADO', 'VALOR_PAGADO']
    for col in columnas_financieras:
        if col in df.columns:
            # Remoción de comas, signos de peso y espacios, forzando la conversión a formato flotante numérico
            # Corrected regex to remove non-digit and non-dot characters
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
    print("✓ Normalización Financiera completada.")

    # 4. INGENIERÍA DE EQUIDAD: Atributo Protegido Estructural (A)
    # Segmentación basada en el Poder de Mercado y tamaño del vehículo
    if 'CONFIGURACION' in df.columns:
        df['configuracion_num'] = pd.to_numeric(df['CONFIGURACION'], errors='coerce')

        # A=0: Camión Sencillo de 2 ejes (Pequeño Transportador / Grupo Protegido)
        # A=1: Tipologías mayores (Flota Corporativa / Grupo Favorecido)
        df['A_estructural'] = np.where(df['configuracion_num'] <= 2, 0, 1)
        print("✓ Atributo Protegido (A) configurado por tipología de flota.")

    # 5. INGENIERÍA DE EQUIDAD: Variable Objetivo Financiera (Y)
    # Identificación del riesgo de ineficiencia comercial o precarización
    if 'VALOR_PACTADO' in df.columns:
        # Eliminación temporal de fletes nulos o en cero para evitar distorsiones estadísticas
        df = df[df['VALOR_PACTADO'] > 0]

        # Cálculo del cuartil inferior (Percentil 25) como límite de precarización
        umbral_sice = df['VALOR_PACTADO'].quantile(0.25)

        # Y=1 si el flete es menor o igual al umbral crítico (Anomalía Tarifaria), de lo contrario Y=0
        df['Y_anomalia_tarifaria'] = (df['VALOR_PACTADO'] <= umbral_sice).astype(int)
        print(f"✓ Variable Objetivo (Y) configurada. Umbral crítico tarifario (Q1): ${umbral_sice:,.2f} COP.")

    # 6. PROTOCOLO DE LIMPIEZA: Integridad de Grupos
    # Imputación técnica o eliminación estratificada para evitar sesgo de selección inicial
    variables_criticas = ['A_estructural', 'Y_anomalia_tarifaria', 'VALOR_PACTADO']
    df_limpio = df.dropna(subset=[col for col in variables_criticas if col in df.columns])

    print("\n=== REPORTE EDA INICIAL (GRUPO 2) ===")
    print(f"Total de registros viables tras limpieza y auditoría financiera: {df_limpio.shape[0]}")

    print("\nDistribución del Atributo Protegido de Flota (A):")
    dist_A = df_limpio['A_estructural'].value_counts(normalize=True) * 100
    print(f"Flota Corporativa / Vehículos Mayores (A=1): {dist_A.get(1, 0):.2f}%")
    print(f"Pequeño Transportador / Camión Sencillo (A=0): {dist_A.get(0, 0):.2f}%")

    print("\nDistribución de la Variable Objetivo Financiera (Y):")
    dist_Y = df_limpio['Y_anomalia_tarifaria'].value_counts(normalize=True) * 100
    print(f"Operación con Flete Eficiente (Y=0): {dist_Y.get(0, 0):.2f}%")
    print(f"Riesgo de Precarización Tarifaria (Y=1): {dist_Y.get(1, 0):.2f}%")

    # Exportación del Dataset Limpio para la Semana 2 (Modelamiento)
    df_limpio.to_csv('RNDC_Grupo2_Limpio_S1.csv', index=False)
    print("\nArchivo 'RNDC_Grupo2_Limpio_S1.csv' exportado exitosamente para la fase de modelamiento algorítmico.")

    return df_limpio
file_path = '/content/drive/MyDrive/Tiempos_Logísticos_de_cada_viaje_de_vehículos_de_carga_20260612.csv'
df_final_g2 = preprocesar_rndc_grupo2(file_path)
display(df_final_g2.head())
print(df_final_g2.columns.tolist())
# Ejecución de la función (Los estudiantes deben ajustar la ruta a su archivo local)
# df_final_g2 = preprocesar_rndc_grupo2('muestra_rndc.csv')

=== INICIANDO PROTOCOLO DE LIMPIEZA RNDC - GRUPO 2 ===
Datos originales cargados: 836013 registros.
✓ Corrección Temporal completada (Desfase de 1899 solucionado).
✓ Normalización Financiera completada.
✓ Atributo Protegido (A) configurado por tipología de flota.
✓ Variable Objetivo (Y) configurada. Umbral crítico tarifario (Q1): $408.00 COP.

=== REPORTE EDA INICIAL (GRUPO 2) ===
Total de registros viables tras limpieza y auditoría financiera: 672785

Distribución del Atributo Protegido de Flota (A):
Flota Corporativa / Vehículos Mayores (A=1): 51.78%
Pequeño Transportador / Camión Sencillo (A=0): 48.22%

Distribución de la Variable Objetivo Financiera (Y):
Operación con Flete Eficiente (Y=0): 74.99%
Riesgo de Precarización Tarifaria (Y=1): 25.01%

Archivo 'RNDC_Grupo2_Limpio_S1.csv' exportado exitosamente para la fase de modelamiento algorítmico.


,AÑO/MES,NATURALEZA,COD_PRODUCTO,PRODUCTO,CANTIDAD,UNID_MEDIDA,FECHASALIDACARGUE,HORA_SALIDA_CARGUE,FECHALLEGADADESCARGUE,HORA_LLEGADA_DESCARGUE,...,VALOR_PAGADO,CANTIDAD_REMESAS_VIAJE,EMPRESA_TRANSPORTE,PLACA,CONFIGURACION,CONDUCTOR,FECHA_HORA_SALIDA_CORREGIDA,configuracion_num,A_estructural,Y_anomalia_tarifaria
0,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,150,Kilogramos,2021 Jan 26 12:00:00 AM,1899-12-31T19:01:00.000,2021 Jan 28 12:00:00 AM,1899-12-31T17:01:00.000,...,260.0,5,"2,293",65211,2,"146,021",2021-01-26 19:01:00,2.0,0,1
1,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,350,Kilogramos,2021 Jan 26 12:00:00 AM,1899-12-31T19:59:00.000,2021 Jan 28 12:00:00 AM,1899-12-31T17:59:00.000,...,260.0,5,"2,293",65211,2,"146,021",2021-01-26 19:59:00,2.0,0,1
2,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40,Kilogramos,2021 Jan 26 12:00:00 AM,1899-12-31T19:10:00.000,2021 Jan 28 12:00:00 AM,1899-12-31T17:10:00.000,...,260.0,5,"2,293",65211,2,"146,021",2021-01-26 19:10:00,2.0,0,1
3,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40,Kilogramos,2021 Jan 26 12:00:00 AM,1899-12-31T19:11:00.000,2021 Jan 28 12:00:00 AM,1899-12-31T17:11:00.000,...,260.0,5,"2,293",65211,2,"146,021",2021-01-26 19:11:00,2.0,0,1
4,202101,Carga Normal,9604,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40,Kilogramos,2021 Jan 26 12:00:00 AM,1899-12-31T19:02:00.000,2021 Jan 28 12:00:00 AM,1899-12-31T17:03:00.000,...,260.0,5,"2,293",65211,2,"146,021",2021-01-26 19:02:00,2.0,0,1


['AÑO/MES', 'NATURALEZA', 'COD_PRODUCTO', 'PRODUCTO', 'CANTIDAD', 'UNID_MEDIDA', 'FECHASALIDACARGUE', 'HORA_SALIDA_CARGUE', 'FECHALLEGADADESCARGUE', 'HORA_LLEGADA_DESCARGUE', 'CODIGO_CARGUE', 'CARGUE', 'CODIGO_DESCARGUE', 'DESCARGUE', 'HORAS_VIAJE', 'HORAS_ESPERA_CARGUE', 'HORAS_CARGUE', 'HORAS_ESPERA_DESCARGUE', 'HORAS_DESCARGUE', 'VALOR_PACTADO', 'VALOR_PAGADO', 'CANTIDAD_REMESAS_VIAJE', 'EMPRESA_TRANSPORTE', 'PLACA', 'CONFIGURACION', 'CONDUCTOR', 'FECHA_HORA_SALIDA_CORREGIDA', 'configuracion_num', 'A_estructural', 'Y_anomalia_tarifaria']


In [5]:
import re
import numpy as np
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/RNDC_Grupo2_Limpio_S1.csv')
def convert_quantity_to_kg(row):

    cantidad = pd.to_numeric(row['CANTIDAD'], errors='coerce')
    unidad = row['UNID_MEDIDA']

    producto = str(row['PRODUCTO']).lower()

    # palabras clave para asignar densidad a los liquidos dependiendo de si aparece keyword en producto,
    # basandose en valores que obtuve con ayuda de notebooklm y cuyo origen esta en densidades.ris
    keyword_to_density = {
        'gasoleo': 0.835,
        'diésel': 0.835,
        'gasolina': 0.75,
        'nafta': 0.75,
        'petróleo bruto': 0.885,
        'crudo': 0.885,
        'destilados de petróleo': 0.765,
        'dipropilcetona': 0.815,
        'clorosilanos': 1.075,
        'hidrocarburos líquidos': 0.75,
        'etanol y gasolina': 0.765,
        'petróleo bruto ácido': 0.885,
    }
#ésta está para en caso de que ninguna keyword este en producto use esta alv
    densidad_promedio = 0.8 #basado en la del agua

    if pd.isna(cantidad):
        return np.nan

    for keyword, density in keyword_to_density.items():
        if keyword in producto:
            densidad_promedio = density
            break

    if unidad == 'Kilogramos':
        return cantidad
    elif unidad == 'Galones':
        gallons_to_liters = 3.78541
        return cantidad * gallons_to_liters * densidad_promedio
    else:
        return np.nan

df['CANTIDAD_KG'] = df.apply(convert_quantity_to_kg, axis=1)

display(df[['CANTIDAD', 'UNID_MEDIDA', 'PRODUCTO', 'CANTIDAD_KG']].head())

df.to_csv('/content/drive/MyDrive/COMPLETORNDC_Grupo2_Limpio_S1.csv', index=False)

,CANTIDAD,UNID_MEDIDA,PRODUCTO,CANTIDAD_KG
0,150.0,Kilogramos,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,150.0
1,350.0,Kilogramos,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,350.0
2,40.0,Kilogramos,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40.0
3,40.0,Kilogramos,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40.0
4,40.0,Kilogramos,MANUFACTURAS DIVERSAS-TAMICES/CEDAZOS Y CRIBAS...,40.0


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer # Import SimpleImputer

def ejecutar_laboratorio_financiero_semana2(ruta_csv):
    print("==========================================================")
    print("SISTEMAS NEURODIFUSOS - AUDITORÍA DE SESGO DE FLOTA (S2)")
    print("==========================================================\n")

    # 1. Carga del dataset financiero de la Semana 1
    try:
        df = pd.read_csv(ruta_csv)
        print(f"✓ Dataset cargado exitosamente. Registros: {df.shape[0]}")
    except Exception as e:
        print(f"Error al cargar el archivo financiero de la Semana 1: {e}")
        return

    # 2. Selección de descriptores del flete (X), Atributo de Flota (A) y Target Tarifario (Y)
    # Se aíslan variables del RNDC que describen la operación comercial
    "NOMBRES DE VARIABLES REEMPLAZADOS POR LAS COLUMNAS REALES EN EL DATASET"
    caracteristicas_comerciales = ['CANTIDAD_REMESAS_VIAJE', 'HORAS_VIAJE', 'CANTIDAD_KG']

    # Si las variables explícitas requieren adaptación, se estructuran en el entorno
    if not all(col in df.columns for col in caracteristicas_comerciales):
        np.random.seed(42)
        df['CANTIDAD_REMESAS_VIAJE'] = np.random.choice([1, 2, 3], size=len(df), p=[0.8, 0.15, 0.05])
        df['HORAS_VIAJE'] = np.random.normal(24, 8, size=len(df)).clip(1, 120)
        df['CANTIDAD_KG'] = np.where(df['A_estructural'] == 0, np.random.normal(8, 2, size=len(df)), np.random.normal(32, 4, size=len(df)))

    X = df[caracteristicas_comerciales]
    y = df['Y_anomalia_tarifaria']
    A = df['A_estructural']

    "llenar valores pq si no no funciona alv foken shet"
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)
    X = pd.DataFrame(X_imputed, columns=X.columns, index=X.index)

    # 3. Partición de datos con preservación de estratificación
    X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
        X, y, A, test_size=0.3, random_state=42, stratify=A
    )

    print(f"Muestras de entrenamiento comercial: {X_train.shape[0]}")
    print(f"Muestras de validación tarifaria:    {X_test.shape[0]}\n")

    # 4. Modelamiento Predictivo Base (Optimización Pura de Mercado)
    modelo_financiero = GradientBoostingClassifier(random_state=42)
    modelo_financiero.fit(X_train, y_train)

    # Inferencia de riesgo
    y_pred = modelo_financiero.predict(X_test)

    # 5. Evaluación del Rendimiento Predictivo
    print("--- RENDIMIENTO GLOBAL DEL MODELO COMERCIAL ---")
    print(f"Accuracy Global: {accuracy_score(y_test, y_pred):.4f}")
    print("\nReporte de Clasificación Comercial:")
    print(classification_report(y_test, y_pred))

    # 6. ENGINE DE AUDITORÍA DE EQUIDAD FINANCIERA
    df_eval = pd.DataFrame({
        'A_flota': A_test.values,
        'Y_real': y_test.values,
        'Y_pred': y_pred
    })

    grupo_protegido = df_eval[df_eval['A_flota'] == 0]   # Pequeño Transportador
    grupo_favorecido = df_eval[df_eval['A_flota'] == 1]  # Flota Corporativa

    print("\n--- RENDIMIENTO POR GRUPO DE FLOTA (A=0: Pequeño Transportador) ---")
    print(classification_report(grupo_protegido['Y_real'], grupo_protegido['Y_pred']))

    print("\n--- RENDIMIENTO POR GRUPO DE FLOTA (A=1: Flota Corporativa) ---")
    print(classification_report(grupo_favorecido['Y_real'], grupo_favorecido['Y_pred']))
    # Tasas de asignación predicha de flete precario
    tasa_prot = grupo_protegido['Y_pred'].mean()
    tasa_fav = grupo_favorecido['Y_pred'].mean()

    # Computación de métricas de justicia distributiva
    spd = tasa_prot - tasa_fav
    di = tasa_prot / tasa_fav if tasa_fav > 0 else np.nan

    def obtener_tpr(df_sub):
        cm = confusion_matrix(df_sub['Y_real'], df_sub['Y_pred'], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_prot = obtener_tpr(grupo_protegido)
    tpr_fav = obtener_tpr(grupo_favorecido)
    eod = tpr_prot - tpr_fav

    # 7. REPORTE CIENTÍFICO DE ASIMETRÍA DE MERCADO
    print("--- INFORME MATEMÁTICO DE SESGO DE FLOTA ---")
    print(f"Predicción de Flete Precario en Pequeño Transportador (A=0): {tasa_prot:.2%}")
    print(f"Predicción de Flete Precario en Flota Corporativa     (A=1): {tasa_fav:.2%}")
    print(f"Diferencia de Paridad Estadística (SPD):                    {spd:.4f}")
    print(f"Impacto Dispar (DI) Tarifario:                              {di:.4f}")
    print("-" * 50)
    print(f"Sensibilidad ante Tarifas Degradadas - Pequeño (A=0):       {tpr_prot:.2%}")
    print(f"Sensibilidad ante Tarifas Degradadas - Corporativo (A=1):    {tpr_fav:.2%}")
    print(f"Diferencia de Igualdad de Oportunidades (EOD):              {eod:.4f}\n")

    print("--- DIAGNÓSTICO DE RIESGO REGULATORIO ---")
    if spd > 0.10:
        print("[ALERTA EXTREMA] El algoritmo normaliza la precarización económica del pequeño propietario.")
    else:
        print("[PASÓ] Distribución de riesgo tarifario balanceada entre flotas.")

    if di > 1.25:
        print("[ALERTA EXTREMA] Impacto Dispar Financiero Detectado. Alta vulnerabilidad frente al SICE-TAC.")
    else:
        print("[PASÓ] Cumplimiento de márgenes de impacto proporcional.")

    if abs(eod) > 0.05:
        print("[ALERTA EXTREMA] El modelo discrimina en la precisión del riesgo de flete deficitario.")
    else:
        print("[PASÓ] Paridad en el reconocimiento de fallas tarifarias de mercado.")
    print("==========================================================")

In [13]:
ejecutar_laboratorio_financiero_semana2('/content/drive/MyDrive/COMPLETORNDC_Grupo2_Limpio_S1.csv')

SISTEMAS NEURODIFUSOS - AUDITORÍA DE SESGO DE FLOTA (S2)

✓ Dataset cargado exitosamente. Registros: 672785
Muestras de entrenamiento comercial: 470949
Muestras de validación tarifaria:    201836

--- RENDIMIENTO GLOBAL DEL MODELO COMERCIAL ---
Accuracy Global: 0.8073

Reporte de Clasificación Comercial:
              precision    recall  f1-score   support

           0       0.83      0.94      0.88    151489
           1       0.70      0.40      0.51     50347

    accuracy                           0.81    201836
   macro avg       0.76      0.67      0.70    201836
weighted avg       0.79      0.81      0.79    201836


--- RENDIMIENTO POR GRUPO DE FLOTA (A=0: Pequeño Transportador) ---
              precision    recall  f1-score   support

           0       0.76      0.91      0.83     66345
           1       0.66      0.37      0.47     30984

    accuracy                           0.74     97329
   macro avg       0.71      0.64      0.65     97329
weighted avg       0.72   

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
